# C02 Lesson01：Token、Token ID、Embedding 与模型输入

**状态：本课实践已完成。** 用户确认完成作业；本次整理保留原有笔记与输入顺序，修正 PAD 检查并重新运行。

原有第 1～6 部分为用户保存的课程实践；第 7～11 部分为助手根据本次作业对照补充的参考实验，不作为独立实现的证据。

运行环境：Conda `fde`、Python 3.11.16、PyTorch 2.14.0、CPU。请选择正确解释器后重启内核、运行全部单元格。

建词表 → 切分编号 → 补齐 → 查表 → 检查结果

[知识点](../docs/C02-Lesson01-notes.md) · [作业与第四题解释](../docs/C02-Lesson01-homework.md) · [交接](../docs/C02-Lesson01-handoff.md)

In [1]:
import torch
from torch import nn

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)


## 1. 教学用词表：列表位置就是 Token ID。

token_to_id 是 Python 字典，负责把 Token 映射为编号。

In [2]:
vocab = ["<PAD>", "<UNK>", "我", "查询", "订单", "生成", "报告","退款","分析",]
token_to_id = {token: idx for idx, token in enumerate(vocab)}
pad_id = token_to_id["<PAD>"]
unk_id = token_to_id["<UNK>"]

## 2. 教学用分词：输入中故意保留空格。

嵌套列表推导式，逐条文本、逐个 Token 地编号。

token_to_id.get(token, unk_id)
如果字典有这个 Token，就取其编号；如果没有，就使用 unk_id。

In [3]:
texts = ["我 查询 订单", "生成 订单 分析 报告 ","查询 退款"]
tokens_batch = [text.split() for text in texts]
ids_batch = [
    [token_to_id.get(token, unk_id) for token in tokens]
    for tokens in tokens_batch
]

## 3. 补齐长度，构造二维整数张量

In [4]:
T = max(len(ids) for ids in ids_batch)
padded_ids = [ids + [pad_id] * (T - len(ids)) for ids in ids_batch]
input_ids = torch.tensor(padded_ids, dtype=torch.long, device="cpu")
valid_mask = input_ids.ne(pad_id)

## 4. 创建 Embedding 表，执行查表

In [5]:
B = len(texts)
V = len(vocab)
D = 6
embedding = nn.Embedding(
    num_embeddings=V,
    embedding_dim=D,
    padding_idx=pad_id,
    device="cpu",
)
x = embedding(input_ids)

## 5. 观察结构、查表关系和补齐位置。

In [6]:
print("tokens_batch:", tokens_batch)
print("ids_batch:", ids_batch)
print("input_ids:\n", input_ids)
print("valid_mask:\n", valid_mask.long())
print("input_ids.shape:", tuple(input_ids.shape))
print("embedding.weight.shape:", tuple(embedding.weight.shape))
print("x.shape:", tuple(x.shape))
print("input_ids.dtype:", input_ids.dtype)
print("x.dtype:", x.dtype)
print("device:", x.device)
print("embedding.weight[2]:", embedding.weight[2].detach())
print("x[0, 0]:", x[0, 0].detach())
print("lookup_equal:", torch.allclose(x[0, 0], embedding.weight[2]))
# 输入顺序和长度改变后，固定下标不一定还是 PAD。
# 按有效位置标记提取当前批次的全部补齐向量。
pad_vectors = x[~valid_mask]
print("PAD vectors:", pad_vectors.detach())
print("PAD vectors are zero:", torch.allclose(
    pad_vectors, torch.zeros_like(pad_vectors)
))

tokens_batch: [['我', '查询', '订单'], ['生成', '订单', '分析', '报告'], ['查询', '退款']]
ids_batch: [[2, 3, 4], [5, 4, 8, 6], [3, 7]]
input_ids:
 tensor([[2, 3, 4, 0],
        [5, 4, 8, 6],
        [3, 7, 0, 0]])
valid_mask:
 tensor([[1, 1, 1, 0],
        [1, 1, 1, 1],
        [1, 1, 0, 0]])
input_ids.shape: (3, 4)
embedding.weight.shape: (9, 6)
x.shape: (3, 4, 6)
input_ids.dtype: torch.int64
x.dtype: torch.float32
device: cpu
embedding.weight[2]: tensor([-0.728, -0.559, -0.769,  0.762,  1.642, -0.160])
x[0, 0]: tensor([-0.728, -0.559, -0.769,  0.762,  1.642, -0.160])
lookup_equal: True
PAD vectors: tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
PAD vectors are zero: True


## 6. 同一个 Embedding 层中，相同 ID 查出相同向量。

In [7]:
repeat_ids = torch.tensor([[2, 2]], dtype=torch.long, device="cpu")
repeat_x = embedding(repeat_ids)
print("same_id_equal:", torch.allclose(repeat_x[0, 0], repeat_x[0, 1]))

same_id_equal: True


## 7. 参考补充：不同位置的同一个 Token

保留当前三条文本的顺序。“查询”分别位于第一条的第二个位置和第三条的第一个位置。两处 ID 相同，在同一张参数表中查出的向量相同。这不表示后续上下文表示必然相同。

In [8]:
print("两处查询的 ID:", input_ids[0, 1].item(), input_ids[2, 0].item())
same_query_vector = torch.allclose(x[0, 1], x[2, 0])
print("不同位置、相同 Token 的查表向量相同:", same_query_vector)

两处查询的 ID: 3 3
不同位置、相同 Token 的查表向量相同: True


## 8. 参考补充：UNK 是未知内容，不是补齐

保持词表不变，单独处理“我 查询 库存”。“库存”没有收录，按本课规则映射到 UNK；该位置仍然有效。单独一条文本不需要为另一批次补齐。

In [9]:
unknown_text = "我 查询 库存"
unknown_tokens = unknown_text.split()
unknown_ids = torch.tensor(
    [[token_to_id.get(token, unk_id) for token in unknown_tokens]],
    dtype=torch.long,
    device="cpu",
)
unknown_mask = unknown_ids.ne(pad_id)
unknown_x = embedding(unknown_ids)
print("unknown_ids:", unknown_ids)
print("unknown_mask:", unknown_mask.long())
print("unknown_x.shape:", tuple(unknown_x.shape))

unknown_ids: tensor([[2, 3, 1]])
unknown_mask: tensor([[1, 1, 1]])
unknown_x.shape: (1, 3, 6)


## 9. 参考补充：第二题 shape 核对

独立创建 B=3、T=5、V=12、D=6 的例子，不覆盖上面的课程实践变量。输入中的具体编号仅用于观察 shape。整张参数表包含 V×D 个数。

In [10]:
shape_input_ids = torch.tensor(
    [[2, 3, 4, 0, 0], [5, 6, 1, 0, 0], [2, 4, 3, 5, 6]],
    dtype=torch.long,
    device="cpu",
)
shape_embedding = nn.Embedding(12, 6, device="cpu")
shape_x = shape_embedding(shape_input_ids)
shape_mask = shape_input_ids.ne(0)
print("input_ids:", tuple(shape_input_ids.shape))
print("embedding.weight:", tuple(shape_embedding.weight.shape))
print("x:", tuple(shape_x.shape))
print("valid_mask:", tuple(shape_mask.shape))
print("x[1]:", tuple(shape_x[1].shape))
print("x[1, 3]:", tuple(shape_x[1, 3].shape))
print("参数数值总量:", shape_embedding.weight.numel())

input_ids: (3, 5)
embedding.weight: (12, 6)
x: (3, 5, 6)
valid_mask: (3, 5)
x[1]: (5, 6)
x[1, 3]: (6,)
参数数值总量: 72


## 10. 参考补充：第四题的两类错误与修正

下面故意触发两类错误，再分别修正。使用 `try/except` 展示预期错误，避免第一个错误中断整本运行。

### A：类型不符合整数索引要求

`2.0` 和 `3.0` 使输入成为浮点张量。Embedding 要求整数索引，本课统一使用 `torch.long`。编号值本身在范围内，修复的是类型。

In [11]:
expected_errors = {}
layer_a = nn.Embedding(7, 4, device="cpu")
bad_ids_a = torch.tensor([[2.0, 3.0]], device="cpu")
print("案例 A 输入类型:", bad_ids_a.dtype)
try:
    layer_a(bad_ids_a)
except RuntimeError as exc:
    expected_errors["A"] = {"type": type(exc).__name__, "message": str(exc)}
    print("预期错误 A:", type(exc).__name__, str(exc))
else:
    raise AssertionError("案例 A 应拒绝浮点索引")

fixed_ids_a = torch.tensor([[2, 3]], dtype=torch.long, device="cpu")
fixed_result_a = layer_a(fixed_ids_a)
print("修正后的类型:", fixed_ids_a.dtype)
print("修正后的输出 shape:", tuple(fixed_result_a.shape))

案例 A 输入类型: torch.float32
预期错误 A: RuntimeError Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)
修正后的类型: torch.int64
修正后的输出 shape: (1, 2, 4)


### B：类型正确，但 ID 超出当前表的范围

这里重新创建的是七行表，合法 ID 为 0～6，ID=7 要访问不存在的第八行。第三题的九行表中 ID=7 可以合法表示“退款”，但不能把那一套编号直接传给这里的七行表。

原题未给出 ID=7 原本想表达的 Token，因此不能唯一恢复意图。下面假设想输入“报告”，根据原七项词表得到编号 6。修正依据是词表映射，不是任意换成一个较小数字。

In [12]:
original_vocab = ["<PAD>", "<UNK>", "我", "查询", "订单", "生成", "报告"]
original_mapping = {token: idx for idx, token in enumerate(original_vocab)}
layer_b = nn.Embedding(len(original_vocab), 4, device="cpu")
bad_ids_b = torch.tensor([[7]], dtype=torch.long, device="cpu")
print("案例 B 表的行数:", layer_b.num_embeddings)
print("案例 B 输入编号:", bad_ids_b)
try:
    layer_b(bad_ids_b)
except IndexError as exc:
    expected_errors["B"] = {"type": type(exc).__name__, "message": str(exc)}
    print("预期错误 B:", type(exc).__name__, str(exc))
else:
    raise AssertionError("案例 B 应拒绝超出表范围的索引")

fixed_ids_b = torch.tensor(
    [[original_mapping["报告"]]], dtype=torch.long, device="cpu"
)
fixed_result_b = layer_b(fixed_ids_b)
print("报告的正确编号:", fixed_ids_b)
print("修正后的输出 shape:", tuple(fixed_result_b.shape))

案例 B 表的行数: 7
案例 B 输入编号: tensor([[7]])
预期错误 B: IndexError index out of range in self
报告的正确编号: tensor([[6]])
修正后的输出 shape: (1, 1, 4)


## 11. 参考补充：本次运行检查

检查本课的 shape、输入映射、PAD、UNK、不同位置查表关系，以及预期错误和修正结果。这些检查验证代码行为，不作为已经训练语言模型或独立掌握知识的证明。

In [13]:
import json

verification_checks = {
    "main_dimensions": (B, T, V, D) == (3, 4, 9, 6),
    "input_ids": input_ids.tolist() == [[2, 3, 4, 0], [5, 4, 8, 6], [3, 7, 0, 0]],
    "input_shape": tuple(input_ids.shape) == (3, 4),
    "embedding_shape": tuple(embedding.weight.shape) == (9, 6),
    "output_shape": tuple(x.shape) == (3, 4, 6),
    "valid_mask": valid_mask.long().tolist() == [[1, 1, 1, 0], [1, 1, 1, 1], [1, 1, 0, 0]],
    "lookup": torch.allclose(x, embedding.weight[input_ids]),
    "pad_vectors_zero": pad_vectors.shape[0] == 3 and torch.allclose(pad_vectors, torch.zeros_like(pad_vectors)),
    "same_token_different_positions": same_query_vector,
    "unknown_id_and_validity": unknown_ids.tolist() == [[2, 3, 1]] and bool(unknown_mask.all()),
    "shape_exercise": tuple(shape_x.shape) == (3, 5, 6) and tuple(shape_x[1].shape) == (5, 6) and tuple(shape_x[1, 3].shape) == (6,) and shape_embedding.weight.numel() == 72,
    "case_a_expected_error": expected_errors["A"]["type"] == "RuntimeError",
    "case_a_fixed": fixed_ids_a.dtype == torch.long and tuple(fixed_result_a.shape) == (1, 2, 4),
    "case_b_expected_error": expected_errors["B"]["type"] == "IndexError",
    "case_b_fixed": fixed_ids_b.item() == original_mapping["报告"] and tuple(fixed_result_b.shape) == (1, 1, 4),
}
print(json.dumps(verification_checks, ensure_ascii=False, indent=2))
assert all(verification_checks.values()), "存在未通过的检查"
print("本次运行检查全部通过；两类原始错误均已按预期触发并修正。")

{
  "main_dimensions": true,
  "input_ids": true,
  "input_shape": true,
  "embedding_shape": true,
  "output_shape": true,
  "valid_mask": true,
  "lookup": true,
  "pad_vectors_zero": true,
  "same_token_different_positions": true,
  "unknown_id_and_validity": true,
  "shape_exercise": true,
  "case_a_expected_error": true,
  "case_a_fixed": true,
  "case_b_expected_error": true,
  "case_b_fixed": true
}
本次运行检查全部通过；两类原始错误均已按预期触发并修正。


## 后续学习入口（2026-10-22 更新）

本 Notebook 是 Lesson01 的历史实践。C02 Lesson01–05 已完成授课与作业答案；本轮模块复核结果见 [C02 总览与验收](../docs/C02-acceptance.md)。

下一模块从 C03 Lesson01：真实 Tokenizer 与模型输入编码开始，C03 尚未授课。详见 [后续课程交接](../docs/C02-to-C03-handoff.md)。

本课仅初始化与查表，不含训练或真实模型 API 调用。
